# 🚁 VisDrone2019-MOT — Method 1 (Baseline): YOLOv11 + SORT## Computer Vision — Assignment IV: Detection and Tracking**Pipeline:** YOLOv11 (detector) → SORT (tracker, purely geometric)  **Dataset:** VisDrone2019-MOT-val  **Classes:** Pedestrian (class 1) & Car (class 4)  **Metrics:** MOTA, IDF1, ID Switches, FPS  > SORT (Simple Online and Realtime Tracking) uses a Kalman Filter for motion prediction  > and the Hungarian algorithm for IoU-based data association. No appearance features.

In [ ]:
# ── 1. Instalación de dependencias ──────────────────────────────!pip install ultralytics filterpy motmetrics lap -q

In [ ]:
# ── 2. Imports y Configuración ──────────────────────────────────import os, glob, time, warningsimport numpy as npimport cv2from pathlib import Pathfrom collections import defaultdictimport matplotlib.pyplot as pltimport matplotlib.patches as patchesfrom IPython.display import display, clear_outputwarnings.filterwarnings("ignore")# ── Rutas Kaggle ──# Ajusta DATASET_ROOT según el nombre de tu dataset en Kaggle Input.DATASET_ROOT = "/kaggle/input/visdrone2019-mot-val/VisDrone2019-MOT-val"SEQ_DIR = os.path.join(DATASET_ROOT, "sequences")ANN_DIR = os.path.join(DATASET_ROOT, "annotations")OUTPUT_DIR = "/kaggle/working/results/yolov11_sort"os.makedirs(OUTPUT_DIR, exist_ok=True)# ── Mapeo de clases ──────────────────────────────────────────────# ⚠️  IMPORTANTE — Tres sistemas de numeración coexisten:##  1. Anotaciones VisDrone-MOT (archivos .txt del GT, 1-indexed):#       1=pedestrian, 2=people, 3=bicycle, 4=car, 5=van, 6=truck...##  2. YOLO entrenado en VisDrone con Ultralytics (0-indexed, ver YAML):#       0=pedestrian, 1=people, 2=bicycle, 3=car, 4=van, 5=truck...#       (Ultralytics hace cls = int(row[5]) - 1 al convertir)##  3. COCO pretrained YOLOv11 (clases COCO estándar, 0-indexed):#       0=person,  2=car,  5=bus,  7=truck ...## Este notebook usa modelo COCO pretrained (yolo11m.pt).# Mapeamos: COCO 0 (person) → VisDrone-MOT 1 (pedestrian)#           COCO 2 (car)    → VisDrone-MOT 4 (car)## ⚠️  LIMITACIÓN CONOCIDA: COCO "person" abarca tanto VisDrone class 1# (pedestrian: caminando/de pie) como class 2 (people: otras posturas).# El GT solo evalúa class 1, por lo que detecciones de class-2 "people"# generarán FP. Para minimizar esto se puede bajar YOLO_CONF.## Si usas un modelo pre-entrenado en VisDrone (no COCO), cambia:#   TARGET_COCO_CLASSES = [0, 3]   # pedestrian=0, car=3 (VisDrone 0-indexed)#   COCO_TO_VISDRONE    = {0: 1, 3: 4}# ────────────────────────────────────────────────────────────────COCO_TO_VISDRONE = {0: 1, 2: 4}TARGET_COCO_CLASSES = list(COCO_TO_VISDRONE.keys())    # [0, 2]TARGET_VISDRONE_CLASSES = set(COCO_TO_VISDRONE.values())  # {1, 4}# ── Hiperparámetros ──YOLO_MODEL    = "yolo11m.pt"  # YOLOv11 medium (balance velocidad/precisión en T4)YOLO_CONF     = 0.25          # Umbral de confianza mínimoYOLO_IOU_NMS  = 0.45          # IoU para NMSYOLO_IMGSZ    = 1280          # Resolución de inferencia (mayor → mejor small objects)SORT_MAX_AGE       = 30   # Frames sin detección antes de eliminar trackSORT_MIN_HITS      = 3    # Detecciones mínimas consecutivas para confirmar trackSORT_IOU_THRESHOLD = 0.3  # Umbral IoU para asociaciónprint(f"Dataset root : {DATASET_ROOT}")print(f"Sequences    : {SEQ_DIR}")print(f"Annotations  : {ANN_DIR}")print(f"Output       : {OUTPUT_DIR}")seqs = sorted([d for d in os.listdir(SEQ_DIR) if os.path.isdir(os.path.join(SEQ_DIR, d))])print(f"\nSecuencias encontradas: {len(seqs)}")for s in seqs:    n_frames = len(glob.glob(os.path.join(SEQ_DIR, s, "*.jpg")))    print(f"  {s}: {n_frames} frames")

## 📂 Utilidades del DatasetFunciones para cargar el Ground Truth y las imágenes de cada secuencia.

In [ ]:
# ── 3. Funciones de carga del dataset ──────────────────────────def load_gt(ann_dir, seq_name, target_classes={1, 4}):    """    Carga las anotaciones Ground Truth de una secuencia VisDrone-MOT.    Formato real de los archivos (10 columnas, ref: paper Zhu et al. TPAMI 2022):      <frame>,<id>,<bb_left>,<bb_top>,<bb_width>,<bb_height>,      <score>,<object_category>,<truncation>,<occlusion>    Nota sobre clases (1-indexed en los archivos):      1=pedestrian, 2=people, 3=bicycle, 4=car, 5=van, 6=truck,      7=tricycle, 8=awning-tricycle, 9=bus, 10=motor    Nota sobre score:      score=0 → región ignorada (no se evalúa, no incluir)      score=1 → objeto real a evaluar    Filtra: score > 0  AND  class in target_classes ({1, 4} por defecto)    """    ann_file = os.path.join(ann_dir, f"{seq_name}.txt")    gt = defaultdict(list)    with open(ann_file, 'r') as f:        for line in f:            parts = line.strip().split(',')            if len(parts) < 8:   # mínimo 8 campos requeridos                continue            frame     = int(parts[0])            obj_id    = int(parts[1])            bb_left   = float(parts[2])            bb_top    = float(parts[3])            bb_width  = float(parts[4])            bb_height = float(parts[5])            score     = float(parts[6])   # 0 = ignorado, 1 = válido            obj_class = int(parts[7])     # 1-indexed (pedestrian=1, car=4)            # parts[8] = truncation, parts[9] = occlusion (opcionales)            # Filtrar regiones ignoradas (score=0) y clases no relevantes            if score > 0 and obj_class in target_classes:                gt[frame].append({                    'id'       : obj_id,                    'bb_left'  : bb_left,                    'bb_top'   : bb_top,                    'bb_width' : bb_width,                    'bb_height': bb_height,                    'class'    : obj_class                })    return gtdef get_frame_paths(seq_dir, seq_name):    """Retorna lista ordenada de rutas a los frames de una secuencia."""    frames_dir = os.path.join(seq_dir, seq_name)    frames = sorted(glob.glob(os.path.join(frames_dir, "*.jpg")))    return frames# ── Verificar carga del GT ──seq_test = seqs[0]gt_test  = load_gt(ANN_DIR, seq_test)n_gt_frames = len(gt_test)n_gt_objs   = sum(len(v) for v in gt_test.values())print(f"Secuencia de prueba: {seq_test}")print(f"  Frames con GT (ped+car): {n_gt_frames}")print(f"  Objetos totales (ped+car): {n_gt_objs}")print(f"  Ejemplo frame 1: {gt_test.get(1, 'Sin objetos')[:3]}")

## 🔧 Implementación de SORT (Simple Online and Realtime Tracking)SORT consta de tres componentes:1. **Kalman Filter:** Modelo de velocidad constante para predecir la posición futura de cada track. El estado es `[cx, cy, s, r, vcx, vcy, vs]` donde `s = area` y `r = aspect ratio`.2. **Hungarian Algorithm:** Asignación óptima entre detecciones y tracks predichos, minimizando el costo basado en IoU.3. **Track Management:** Creación, actualización y eliminación de tracks según hits y age.> Referencia: Bewley et al., "Simple Online and Realtime Tracking" (ICIP 2016)

In [ ]:
# ── 4. Implementación de SORT ──────────────────────────────────from filterpy.kalman import KalmanFilterfrom scipy.optimize import linear_sum_assignmentdef convert_bbox_to_z(bbox):    """    Convierte bounding box de formato [x1,y1,x2,y2] a estado de observación [cx,cy,s,r].    - cx, cy: centro del bbox    - s: escala (área)    - r: aspect ratio (w/h)    """    w = bbox[2] - bbox[0]    h = bbox[3] - bbox[1]    cx = bbox[0] + w / 2.0    cy = bbox[1] + h / 2.0    s = w * h          # área    r = w / (h + 1e-6)  # aspect ratio    return np.array([cx, cy, s, r]).reshape((4, 1))def convert_x_to_bbox(x):    """    Convierte estado del Kalman [cx,cy,s,r,...] a bounding box [x1,y1,x2,y2].    """    w = np.sqrt(np.maximum(x[2] * x[3], 0))  # sqrt(s * r)    h = x[2] / (w + 1e-6)                     # s / w    return np.array([        x[0] - w / 2.0,  # x1        x[1] - h / 2.0,  # y1        x[0] + w / 2.0,  # x2        x[1] + h / 2.0   # y2    ]).flatten()def iou_batch(bb_dets, bb_trks):    """    Calcula la matriz IoU entre dos conjuntos de bounding boxes.    Ambos en formato [x1, y1, x2, y2].        Args:        bb_dets: (N, 4) detecciones        bb_trks: (M, 4) tracks predichos    Returns:        iou_matrix: (N, M)    """    bb_dets = np.atleast_2d(bb_dets)    bb_trks = np.atleast_2d(bb_trks)        # Expandir dimensiones para broadcasting: (N,1,4) vs (1,M,4)    dets = np.expand_dims(bb_dets, 1)    trks = np.expand_dims(bb_trks, 0)        xx1 = np.maximum(dets[..., 0], trks[..., 0])    yy1 = np.maximum(dets[..., 1], trks[..., 1])    xx2 = np.minimum(dets[..., 2], trks[..., 2])    yy2 = np.minimum(dets[..., 3], trks[..., 3])        w = np.maximum(0.0, xx2 - xx1)    h = np.maximum(0.0, yy2 - yy1)    intersection = w * h        area_det = (dets[..., 2] - dets[..., 0]) * (dets[..., 3] - dets[..., 1])    area_trk = (trks[..., 2] - trks[..., 0]) * (trks[..., 3] - trks[..., 1])        iou = intersection / (area_det + area_trk - intersection + 1e-6)    return ioudef associate_detections_to_trackers(detections, trackers, iou_threshold=0.3):    """    Asocia detecciones a trackers usando el algoritmo Húngaro sobre la matriz IoU.        Args:        detections: (N, 4) bboxes de detecciones [x1,y1,x2,y2]        trackers: (M, 4) bboxes predichos de tracks [x1,y1,x2,y2]        iou_threshold: umbral mínimo de IoU para considerar una asociación válida            Returns:        matches: (K, 2) pares [det_idx, trk_idx]        unmatched_detections: índices de detecciones sin match        unmatched_trackers: índices de trackers sin match    """    if len(trackers) == 0:        return (np.empty((0, 2), dtype=int),                np.arange(len(detections)),                np.empty((0,), dtype=int))        if len(detections) == 0:        return (np.empty((0, 2), dtype=int),                np.empty((0,), dtype=int),                np.arange(len(trackers)))        # Calcular matriz IoU (N x M)    iou_matrix = iou_batch(detections, trackers)        # Algoritmo Húngaro (minimiza costo, por eso negamos IoU)    row_indices, col_indices = linear_sum_assignment(-iou_matrix)    matched_indices = np.column_stack((row_indices, col_indices))        # Identificar detecciones y trackers sin match    unmatched_detections = [d for d in range(len(detections)) if d not in matched_indices[:, 0]]    unmatched_trackers = [t for t in range(len(trackers)) if t not in matched_indices[:, 1]]        # Filtrar matches con IoU bajo el umbral    matches = []    for m in matched_indices:        if iou_matrix[m[0], m[1]] < iou_threshold:            unmatched_detections.append(m[0])            unmatched_trackers.append(m[1])        else:            matches.append(m)        matches = np.array(matches).reshape(-1, 2) if matches else np.empty((0, 2), dtype=int)    return matches, np.array(unmatched_detections), np.array(unmatched_trackers)class KalmanBoxTracker:    """    Tracker individual basado en Kalman Filter para un solo objeto.        Estado: [cx, cy, s, r, v_cx, v_cy, v_s]      - (cx, cy): centro del bbox      - s: escala (área del bbox)      - r: aspect ratio (constante en el modelo)      - v_*: velocidades correspondientes          Observación: [cx, cy, s, r]    """    count = 0  # Contador global de IDs        def __init__(self, bbox):        """Inicializa un tracker con el bounding box [x1, y1, x2, y2]."""        # Kalman Filter: 7 estados, 4 observaciones        self.kf = KalmanFilter(dim_x=7, dim_z=4)                # Matriz de transición de estado (modelo de velocidad constante)        self.kf.F = np.array([            [1, 0, 0, 0, 1, 0, 0],  # cx += v_cx            [0, 1, 0, 0, 0, 1, 0],  # cy += v_cy            [0, 0, 1, 0, 0, 0, 1],  # s  += v_s            [0, 0, 0, 1, 0, 0, 0],  # r  (constante)            [0, 0, 0, 0, 1, 0, 0],  # v_cx            [0, 0, 0, 0, 0, 1, 0],  # v_cy            [0, 0, 0, 0, 0, 0, 1],  # v_s        ], dtype=np.float64)                # Matriz de observación        self.kf.H = np.array([            [1, 0, 0, 0, 0, 0, 0],            [0, 1, 0, 0, 0, 0, 0],            [0, 0, 1, 0, 0, 0, 0],            [0, 0, 0, 1, 0, 0, 0],        ], dtype=np.float64)                # Ruido de medición (mayor incertidumbre en s y r)        self.kf.R[2:, 2:] *= 10.0        # Covarianza inicial alta para velocidades (desconocidas)        self.kf.P[4:, 4:] *= 1000.0        self.kf.P *= 10.0        # Ruido del proceso        self.kf.Q[-1, -1] *= 0.01        self.kf.Q[4:, 4:] *= 0.01                # Estado inicial desde el bbox        self.kf.x[:4] = convert_bbox_to_z(bbox)                self.time_since_update = 0        self.id = KalmanBoxTracker.count        KalmanBoxTracker.count += 1        self.hits = 0        self.hit_streak = 0        self.age = 0        def update(self, bbox):        """Actualiza el estado con una nueva detección [x1,y1,x2,y2]."""        self.time_since_update = 0        self.hits += 1        self.hit_streak += 1        self.kf.update(convert_bbox_to_z(bbox))        def predict(self):        """Avanza el estado un paso temporal y retorna el bbox predicho."""        # Evitar áreas negativas        if (self.kf.x[6] + self.kf.x[2]) <= 0:            self.kf.x[6] *= 0.0        self.kf.predict()        self.age += 1        if self.time_since_update > 0:            self.hit_streak = 0        self.time_since_update += 1        return convert_x_to_bbox(self.kf.x)        def get_state(self):        """Retorna el bbox actual estimado [x1,y1,x2,y2]."""        return convert_x_to_bbox(self.kf.x)class SORT:    """    SORT: Simple Online and Realtime Tracking.        Combina:    - Kalman Filter para predicción de movimiento    - Algoritmo Húngaro para asociación por IoU    - Gestión de tracks (creación, confirmación, eliminación)        Args:        max_age: frames máximos sin detección antes de eliminar un track        min_hits: detecciones mínimas consecutivas para confirmar un track        iou_threshold: umbral IoU para la asociación    """    def __init__(self, max_age=30, min_hits=3, iou_threshold=0.3):        self.max_age = max_age        self.min_hits = min_hits        self.iou_threshold = iou_threshold        self.trackers = []        self.frame_count = 0        def update(self, dets=np.empty((0, 5))):        """        Ejecuta un ciclo de tracking: predict → associate → update.                Args:            dets: (N, 5) detecciones [x1, y1, x2, y2, confidence]                    Returns:            tracks: (M, 5) tracks activos [x1, y1, x2, y2, track_id]        """        self.frame_count += 1                # ── PASO 1: Predecir posiciones de tracks existentes ──        predicted_trks = []        to_del = []        for t, trk in enumerate(self.trackers):            pos = trk.predict()            predicted_trks.append(pos)            if np.any(np.isnan(pos)):                to_del.append(t)                # Eliminar tracks con predicción NaN        for t in reversed(to_del):            self.trackers.pop(t)            predicted_trks.pop(t)                predicted_trks = np.array(predicted_trks).reshape(-1, 4) if predicted_trks else np.empty((0, 4))                # ── PASO 2: Asociar detecciones a tracks (Húngaro + IoU) ──        det_bboxes = dets[:, :4] if len(dets) > 0 else np.empty((0, 4))        matched, unmatched_dets, unmatched_trks = associate_detections_to_trackers(            det_bboxes, predicted_trks, self.iou_threshold        )                # ── PASO 3: Actualizar tracks asociados ──        for m in matched:            self.trackers[m[1]].update(dets[m[0], :4])                # ── PASO 4: Crear nuevos tracks para detecciones sin match ──        for i in unmatched_dets:            self.trackers.append(KalmanBoxTracker(dets[i, :4]))                # ── PASO 5: Retornar tracks activos y eliminar muertos ──        ret = []        i = len(self.trackers)        for trk in reversed(self.trackers):            bbox = trk.get_state()            i -= 1            # Solo retornar tracks confirmados (suficientes hits o inicio)            if (trk.time_since_update < 1) and \               (trk.hit_streak >= self.min_hits or self.frame_count <= self.min_hits):                ret.append(np.concatenate((bbox, [trk.id + 1])))  # ID empieza en 1            # Eliminar tracks muertos            if trk.time_since_update > self.max_age:                self.trackers.pop(i)                return np.array(ret).reshape(-1, 5) if ret else np.empty((0, 5))print("✅ SORT implementado correctamente")

## 🔍 Detector YOLOv11Se usa YOLOv11 pre-entrenado en COCO como detector.  Solo se retienen las clases **person** (COCO 0 → VisDrone 1) y **car** (COCO 2 → VisDrone 4).

In [ ]:
# ── 5. Inicialización del detector YOLOv11 ─────────────────────from ultralytics import YOLOmodel = YOLO(YOLO_MODEL)print(f"Modelo cargado: {YOLO_MODEL}")print(f"  Confianza mínima: {YOLO_CONF}")print(f"  IoU NMS: {YOLO_IOU_NMS}")print(f"  Resolución: {YOLO_IMGSZ}")def detect_frame(model, frame, conf=YOLO_CONF, iou=YOLO_IOU_NMS, imgsz=YOLO_IMGSZ):    """    Ejecuta YOLOv11 sobre un frame y retorna detecciones filtradas.        Args:        model: modelo YOLO cargado        frame: imagen BGR (numpy array)            Returns:        dict con keys por clase COCO:            {0: np.array(N, 5) [x1,y1,x2,y2,conf],             2: np.array(M, 5) [x1,y1,x2,y2,conf]}    """    results = model(frame, conf=conf, iou=iou, imgsz=imgsz,                    verbose=False, device=0, classes=TARGET_COCO_CLASSES)        detections_by_class = {cls: [] for cls in TARGET_COCO_CLASSES}        for r in results:        boxes = r.boxes        if boxes is None or len(boxes) == 0:            continue        for box in boxes:            cls_id = int(box.cls[0].item())            if cls_id in TARGET_COCO_CLASSES:                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()                conf_val = float(box.conf[0].item())                detections_by_class[cls_id].append([x1, y1, x2, y2, conf_val])        # Convertir a numpy arrays    for cls_id in detections_by_class:        dets = detections_by_class[cls_id]        detections_by_class[cls_id] = np.array(dets) if dets else np.empty((0, 5))        return detections_by_class# ── Test rápido ──test_frame_path = get_frame_paths(SEQ_DIR, seqs[0])[0]test_frame = cv2.imread(test_frame_path)test_dets = detect_frame(model, test_frame)print(f"\nTest de detección en {os.path.basename(test_frame_path)}:")print(f"  Pedestrians detectados: {len(test_dets[0])}")print(f"  Cars detectados: {len(test_dets[2])}")

## 🔄 Pipeline Modular: Detección → Asociación → TrackingEl pipeline para cada secuencia:1. **Detección:** YOLOv11 detecta pedestrians y cars en cada frame2. **Separación por clase:** Se mantienen trackers independientes por clase para evitar confusión de identidad entre clases3. **Tracking:** Cada SORT tracker asocia detecciones a tracks existentes por IoU4. **Salida MOT:** Resultados en formato MOT estándar

In [ ]:
# ── 6. Pipeline de tracking por secuencia ──────────────────────def run_sequence(model, seq_dir, seq_name, sort_params=None):    """    Ejecuta el pipeline completo YOLOv11 + SORT sobre una secuencia.        Usa trackers SORT separados por clase para mantener identidades    consistentes y evitar confusión cross-clase.        Args:        model: modelo YOLO cargado        seq_dir: directorio de secuencias        seq_name: nombre de la secuencia        sort_params: dict con parámetros de SORT            Returns:        results: list of dicts en formato MOT        fps: frames por segundo del pipeline completo    """    if sort_params is None:        sort_params = {            'max_age': SORT_MAX_AGE,            'min_hits': SORT_MIN_HITS,            'iou_threshold': SORT_IOU_THRESHOLD        }        frames = get_frame_paths(seq_dir, seq_name)        # Resetear contador de IDs global    KalmanBoxTracker.count = 0        # Trackers separados por clase (evita confusión de identidad)    tracker_ped = SORT(**sort_params)   # Pedestrians (COCO 0 → VisDrone 1)    tracker_car = SORT(**sort_params)   # Cars (COCO 2 → VisDrone 4)        results = []    total_time = 0.0        for i, frame_path in enumerate(frames):        frame_idx = i + 1  # MOT format: frames empiezan en 1        frame = cv2.imread(frame_path)                t_start = time.time()                # ── Fase 1: Detección ──        dets_by_class = detect_frame(model, frame)                # ── Fase 2: Tracking por clase ──        ped_tracks = tracker_ped.update(dets_by_class[0])  # person        car_tracks = tracker_car.update(dets_by_class[2])  # car                t_end = time.time()        total_time += (t_end - t_start)                # ── Fase 3: Guardar resultados en formato MOT ──        # Pedestrians (VisDrone class 1)        for trk in ped_tracks:            x1, y1, x2, y2, track_id = trk            results.append({                'frame': frame_idx,                'id': int(track_id),                'bb_left': x1,                'bb_top': y1,                'bb_width': x2 - x1,                'bb_height': y2 - y1,                'conf': 1.0,                'class': 1,                'vis': -1            })                # Cars (VisDrone class 4) — offset de ID para evitar colisión        ID_OFFSET_CAR = 100000        for trk in car_tracks:            x1, y1, x2, y2, track_id = trk            results.append({                'frame': frame_idx,                'id': int(track_id) + ID_OFFSET_CAR,                'bb_left': x1,                'bb_top': y1,                'bb_width': x2 - x1,                'bb_height': y2 - y1,                'conf': 1.0,                'class': 4,                'vis': -1            })                # Progreso        if (i + 1) % 50 == 0 or (i + 1) == len(frames):            elapsed_fps = (i + 1) / total_time if total_time > 0 else 0            print(f"  [{seq_name}] Frame {i+1}/{len(frames)} | "                  f"Peds: {len(ped_tracks)} | Cars: {len(car_tracks)} | "                  f"FPS: {elapsed_fps:.1f}", end="\r")        fps = len(frames) / total_time if total_time > 0 else 0    print(f"  [{seq_name}] Completado: {len(frames)} frames | "          f"Tracks generados: {len(results)} | FPS: {fps:.1f}      ")        return results, fpsdef save_mot_results(results, output_path):    """    Guarda resultados de tracking en formato MOT estándar.        Formato: <frame>,<id>,<bb_left>,<bb_top>,<bb_width>,<bb_height>,<conf>,<class>,<vis>    """    with open(output_path, 'w') as f:        for r in sorted(results, key=lambda x: (x['frame'], x['id'])):            f.write(f"{r['frame']},{r['id']},"                    f"{r['bb_left']:.2f},{r['bb_top']:.2f},"                    f"{r['bb_width']:.2f},{r['bb_height']:.2f},"                    f"{r['conf']:.2f},{r['class']},{r['vis']}\n")print("✅ Pipeline definido")

## 🚀 Ejecución del PipelineProcesamos **todas las secuencias** del VisDrone2019-MOT-val.

In [ ]:
# ── 7. Ejecutar pipeline en todas las secuencias ───────────────all_results = {}all_fps = {}print("=" * 70)print("BASELINE: YOLOv11 + SORT")print("=" * 70)for seq_name in seqs:    print(f"\n📹 Procesando: {seq_name}")    results, fps = run_sequence(model, SEQ_DIR, seq_name)        # Guardar resultados MOT    output_path = os.path.join(OUTPUT_DIR, f"{seq_name}.txt")    save_mot_results(results, output_path)        all_results[seq_name] = results    all_fps[seq_name] = fps    print(f"  💾 Guardado en: {output_path}")avg_fps = np.mean(list(all_fps.values()))print(f"\n{'=' * 70}")print(f"FPS promedio del pipeline: {avg_fps:.2f}")print(f"Resultados guardados en: {OUTPUT_DIR}")print(f"{'=' * 70}")

## 📊 Evaluación con motmetricsMétricas reportadas:- **MOTA** (Multiple Object Tracking Accuracy): Error acumulado de FP, FN e ID switches- **IDF1** (ID F1-score): Capacidad de mantener identidad a largo plazo- **IDS** (ID Switches): Veces que un objeto cambia de identificador- **FPS**: Rendimiento temporal del pipeline completo

In [ ]:
# ── 8. Evaluación cuantitativa ──────────────────────────────────import motmetrics as mmdef evaluate_sequence(gt_dict, pred_list, iou_threshold=0.5):    """    Evalúa una secuencia usando motmetrics.        Args:        gt_dict: dict {frame: [list of gt objects]}        pred_list: list of prediction dicts        iou_threshold: umbral IoU para considerar un match TP            Returns:        acc: MOTAccumulator con los resultados    """    acc = mm.MOTAccumulator(auto_id=True)        # Organizar predicciones por frame    pred_by_frame = defaultdict(list)    for p in pred_list:        pred_by_frame[p['frame']].append(p)        # Obtener todos los frames (unión de GT y predicciones)    all_frames = sorted(set(list(gt_dict.keys()) + list(pred_by_frame.keys())))        for frame_id in all_frames:        gt_frame = gt_dict.get(frame_id, [])        pred_frame = pred_by_frame.get(frame_id, [])                # IDs        gt_ids = [g['id'] for g in gt_frame]        pred_ids = [p['id'] for p in pred_frame]                # Bounding boxes en formato [x, y, w, h] para motmetrics        gt_boxes = np.array([[g['bb_left'], g['bb_top'], g['bb_width'], g['bb_height']]                             for g in gt_frame]) if gt_frame else np.empty((0, 4))        pred_boxes = np.array([[p['bb_left'], p['bb_top'], p['bb_width'], p['bb_height']]                               for p in pred_frame]) if pred_frame else np.empty((0, 4))                # Matriz de distancias basada en IoU        # mm.distances.iou_matrix retorna distancia=1-IoU.        # Internamente hace: dist[IoU < max_iou] = NaN        # → max_iou es el umbral MÍNIMO de IoU para match válido.        # Con max_iou=iou_threshold=0.5: solo se aceptan matches con IoU ≥ 0.5        distances = mm.distances.iou_matrix(gt_boxes, pred_boxes, max_iou=iou_threshold)                acc.update(gt_ids, pred_ids, distances)        return acc# ── Evaluar todas las secuencias ──print("Evaluando todas las secuencias...")print("-" * 70)accumulators = []seq_names = []for seq_name in seqs:    # Cargar GT    gt_dict = load_gt(ANN_DIR, seq_name, target_classes=TARGET_VISDRONE_CLASSES)    # Cargar predicciones    pred_list = all_results[seq_name]        # Evaluar    acc = evaluate_sequence(gt_dict, pred_list)    accumulators.append(acc)    seq_names.append(seq_name)# ── Calcular métricas por secuencia y promedio ──mh = mm.metrics.create()# Métricas por secuenciasummary = mh.compute_many(    accumulators,    metrics=['mota', 'idf1', 'num_switches', 'num_false_positives',             'num_misses', 'precision', 'recall', 'mostly_tracked',             'mostly_lost', 'num_fragmentations'],    names=seq_names,    generate_overall=True  # Agrega fila "OVERALL" al final)# Agregar columna FPSfps_values = [all_fps[s] for s in seq_names] + [avg_fps]summary['fps'] = fps_values# Renombrar columnas para claridadsummary = summary.rename(columns={    'mota': 'MOTA',    'idf1': 'IDF1',    'num_switches': 'IDS',    'num_false_positives': 'FP',    'num_misses': 'FN',    'precision': 'Precision',    'recall': 'Recall',    'mostly_tracked': 'MT',    'mostly_lost': 'ML',    'num_fragmentations': 'Frag',    'fps': 'FPS'})print("\n" + "=" * 70)print("RESULTADOS: YOLOv11 + SORT (Baseline)")print("=" * 70)print(summary.to_string(float_format=lambda x: f"{x:.4f}" if abs(x) < 100 else f"{x:.1f}"))# ── Resumen compacto ──overall = summary.loc['OVERALL']print(f"\n{'=' * 70}")print(f"  📊 RESUMEN OVERALL")print(f"{'=' * 70}")print(f"  MOTA:         {overall['MOTA']:.4f} ({overall['MOTA']*100:.2f}%)")print(f"  IDF1:         {overall['IDF1']:.4f} ({overall['IDF1']*100:.2f}%)")print(f"  ID Switches:  {int(overall['IDS'])}")print(f"  FP:           {int(overall['FP'])}")print(f"  FN:           {int(overall['FN'])}")print(f"  Precision:    {overall['Precision']:.4f}")print(f"  Recall:       {overall['Recall']:.4f}")print(f"  FPS:          {overall['FPS']:.2f}")print(f"{'=' * 70}")

## 🎬 Visualización de ResultadosVisualización de frames con bounding boxes de tracking para inspección cualitativa.

In [ ]:
# ── 9. Visualización de resultados ─────────────────────────────def visualize_tracking(seq_dir, seq_name, pred_list, gt_dict=None,                       frame_indices=None, max_frames=6):    """    Visualiza frames con bounding boxes de tracking.    Verde: pedestrian, Azul: car.    Rojo punteado: Ground Truth (si se provee).    """    frames = get_frame_paths(seq_dir, seq_name)        if frame_indices is None:        # Seleccionar frames distribuidos uniformemente        step = max(1, len(frames) // max_frames)        frame_indices = list(range(0, len(frames), step))[:max_frames]        # Organizar predicciones por frame    pred_by_frame = defaultdict(list)    for p in pred_list:        pred_by_frame[p['frame']].append(p)        n_cols = min(3, len(frame_indices))    n_rows = (len(frame_indices) + n_cols - 1) // n_cols    fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 5 * n_rows))    if n_rows == 1 and n_cols == 1:        axes = np.array([axes])    axes = np.atleast_2d(axes)        colors_ped = {}    colors_car = {}    np.random.seed(42)        for idx, frame_i in enumerate(frame_indices):        row, col = idx // n_cols, idx % n_cols        ax = axes[row, col]                frame_idx = frame_i + 1        frame = cv2.imread(frames[frame_i])        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)        ax.imshow(frame_rgb)                # Dibujar predicciones        for p in pred_by_frame.get(frame_idx, []):            tid = p['id']            cls = p['class']                        if cls == 1:  # Pedestrian                if tid not in colors_ped:                    colors_ped[tid] = np.random.rand(3)                color = colors_ped[tid]                label = f"P-{tid}"            else:  # Car                if tid not in colors_car:                    colors_car[tid] = np.random.rand(3)                color = colors_car[tid]                label = f"C-{tid % 100000}"                        rect = patches.Rectangle(                (p['bb_left'], p['bb_top']), p['bb_width'], p['bb_height'],                linewidth=2, edgecolor=color, facecolor='none'            )            ax.add_patch(rect)            ax.text(p['bb_left'], p['bb_top'] - 4, label,                    fontsize=6, color='white',                    bbox=dict(boxstyle='round,pad=0.15', facecolor=color, alpha=0.8))                # Dibujar GT si existe (rojo punteado)        if gt_dict:            for g in gt_dict.get(frame_idx, []):                rect = patches.Rectangle(                    (g['bb_left'], g['bb_top']), g['bb_width'], g['bb_height'],                    linewidth=1, edgecolor='red', facecolor='none', linestyle='--', alpha=0.5                )                ax.add_patch(rect)                n_preds = len(pred_by_frame.get(frame_idx, []))        ax.set_title(f"Frame {frame_idx} | {n_preds} tracks", fontsize=10)        ax.axis('off')        # Ocultar ejes vacíos    for idx in range(len(frame_indices), n_rows * n_cols):        row, col = idx // n_cols, idx % n_cols        axes[row, col].axis('off')        fig.suptitle(f"YOLOv11 + SORT — {seq_name}", fontsize=14, fontweight='bold')    plt.tight_layout()    plt.show()# ── Visualizar primera y última secuencia ──for seq_name in [seqs[0], seqs[-1]]:    gt_dict = load_gt(ANN_DIR, seq_name)    visualize_tracking(SEQ_DIR, seq_name, all_results[seq_name], gt_dict)

## 💾 Exportar ResumenGuarda las métricas y el resumen para comparación con otros métodos.

In [ ]:
# ── 10. Exportar resumen de métricas ───────────────────────────import json# Guardar métricas en CSVmetrics_path = "/kaggle/working/metrics_yolov11_sort.csv"summary.to_csv(metrics_path)print(f"Métricas guardadas en: {metrics_path}")# Guardar configuración del experimentoconfig = {    "method": "Method 1 (Baseline)",    "detector": YOLO_MODEL,    "tracker": "SORT",    "detector_params": {        "conf": YOLO_CONF,        "iou_nms": YOLO_IOU_NMS,        "imgsz": YOLO_IMGSZ    },    "tracker_params": {        "max_age": SORT_MAX_AGE,        "min_hits": SORT_MIN_HITS,        "iou_threshold": SORT_IOU_THRESHOLD    },    "classes": {"1": "pedestrian", "4": "car"},    "results": {        "MOTA": float(overall['MOTA']),        "IDF1": float(overall['IDF1']),        "IDS": int(overall['IDS']),        "FPS": float(overall['FPS'])    }}config_path = "/kaggle/working/config_yolov11_sort.json"with open(config_path, 'w') as f:    json.dump(config, f, indent=2)print(f"Configuración guardada en: {config_path}")# ── Listar archivos de resultados MOT ──print(f"\nArchivos MOT generados:")for f in sorted(os.listdir(OUTPUT_DIR)):    fpath = os.path.join(OUTPUT_DIR, f)    size = os.path.getsize(fpath)    print(f"  {f}: {size / 1024:.1f} KB")